<a href="https://colab.research.google.com/github/pratishthanagulwar/GenAI/blob/main/GenAI_Prac_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers

In [2]:
import transformers

In [5]:
text= "I like to eat apple . apple is fruit for which prople use to qote 'An apple a day keeps doctor awary'"
print(text)

I like to eat apple . apple is fruit for which prople use to qote 'An apple a day keeps doctor awary'


In [12]:
from transformers import pipeline
classifier = pipeline("text-classification")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [13]:
output = classifier(text)


In [15]:
import pandas as pd
pd.DataFrame(output)

,label,score
0,POSITIVE,0.669669


In [17]:
ntext = " i hate non vegetatrian food as it smells very bad."
noutputs = classifier(ntext)
pd.DataFrame(noutputs)

,label,score
0,NEGATIVE,0.998693


In [18]:
ner_tagger = pipeline("ner", aggregation_strategy="simple")
outputs = ner_tagger(text)
pd.DataFrame(outputs)

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.33GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

""


In [21]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
model_name = "t5-small"
model= AutoModelForSeq2SeqLM.from_pretrained(model_name)
tok = AutoTokenizer.from_pretrained (model_name)
input_text = "summarize: "+ text
inputs = tok(input_text, return_tensors="pt", max_length=512, truncation=True)
summary_ids = model.generate(
    inputs ["input_ids"],
    max_length=56,
    min_length=10,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)
summary = tok.decode(summary_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
print([{"Summary_text": summary}])

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

[{'Summary_text': "apple is fruit for which prople use to qote 'An apple a day keeps doctor awary'"}]


In [24]:
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
qa_model_name = "distilbert-base-cased-distilled-squad"
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
question = "What keeps the doctor away?"
context = text
inputs = qa_tokenizer (question, context, return_tensors="pt")
with torch.no_grad():
  outputs = qa_model (**inputs)
answer_start = torch.argmax(outputs.start_logits)
answer_end = torch.argmax(outputs.end_logits) + 1
answer = qa_tokenizer.convert_tokens_to_string(
    qa_tokenizer.convert_ids_to_tokens (inputs ["input_ids"] [0] [answer_start:answer_end])
)

print(f"Question: (question)")
print(f"Answer: {answer}")

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Question: (question)
Answer: An apple
